<a href="https://colab.research.google.com/github/iabbey-commits/Financial-Research-Agent/blob/main/Agent2_RAG_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

AI Agent 2: RAG

This is an agent that is built to enhances large language models (LLMs) by combining with real-time external data retrieval.

The agent implements a RAG architecture by retrieving PDF documents, extracting text from the document into chunks, embedding the chunks, and stores them in a vector database. It also retrieves relevant information semantically and generates context-aware answers using a Large Language Model (LLM).
The agent utilizes PostgreSQL with pgvector for vector storage, Transformer-based embeddings, and semantic similarity search to obtain context-aware answers derived from the content of the document.
It serves as a knowledge base and reasoning layer, producing responses in the retrieved data.
The main purpose of the agent is to answer user questions using information retrieved directly from uploaded PDF documents instead of using pretrained LLM knowledge.


Below is the archecture and codes for the agent.

In [ ]:
from graphviz import Digraph

def RAG_architecture():
    agent2 = Digraph(format='png')
    agent2.attr(rankdir='TB', size='10,15')

    # Global node style
    agent2.attr('node', shape='rectangle', style='filled', color='lightblue', fontname='Helvetica')

    # Nodes
    agent2.node('A', 'User Query')

    agent2.node('B', '''DocumentQA
- Main Ochestrator ''')

    agent2.node('C', 'PDF Loader \n(PyPdfLoader)')

    agent2.node('D', '''Text splitters
(RecursiveCharacter)''')

    agent2.node('E', '''Embedding Model
(MinniLM-HF)''')

    agent2.node('F', '''PGVector Database
(PostgreSQL + Vector Store)''')

    agent2.node('G', 'Retrieved Context Chunk')

    agent2.node('H', '''Propmt Construction Layer
(Context + User Query)''')
    agent2.node('I', '''Groq LLM Layer
( llama-3.3-70b-versatile Model)''')
    agent2.node('J', 'Final Output')

    # Edges
    agent2.edge('A', 'B', label='Query')
    agent2.edge('B', 'C')
    agent2.edge('B', 'D')
    agent2.edge('B', 'E')
    agent2.edge('C', 'F', label='Raw Document Pages')
    agent2.edge('D', 'F', label='Chunked Documents')
    agent2.edge('E', 'F', label='Vector Embeddings')
    agent2.edge('F', 'G', label='Similarity Search (Top-K)')
    agent2.edge('G', 'H')
    agent2.edge('H', 'I')
    agent2.edge('I', 'J', label='Final Output')


    return agent2


# Generate and save diagram
diagram = RAG_architecture()
diagram.render('RAG_architecture', view=True)


In [ ]:
from IPython.display import Image
Image('RAG_architecture.png')

In [ ]:
# Core installs (quiet + compatible)
!pip install -q \
requests==2.32.4 \
psycopg \
pgvector \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
sentence-transformers \
pypdf \
agno \
groq \
sqlalchemy

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
from google.colab import userdata
userdata.get('MY_HF_TOKEN')

In [ ]:
# Install PostgreSQL dependencies and build tools
!apt-get -qq update
!apt-get -qq install postgresql postgresql-contrib build-essential git postgresql-server-dev-14

# Manual installation of pgvector extension
# Clone, compile, and install pgvector
!git clone --branch v0.6.0 https://github.com/pgvector/pgvector.git
!cd pgvector && PG_CONFIG=/usr/lib/postgresql/14/bin/pg_config make
!cd pgvector && PG_CONFIG=/usr/lib/postgresql/14/bin/pg_config make install

# Start PostgreSQL
!service postgresql start

# Setup database
!sudo -u postgres psql -c "CREATE USER ai WITH PASSWORD 'ai';"
!sudo -u postgres psql -c "CREATE DATABASE ai;"
!sudo -u postgres psql -c "ALTER ROLE ai SET client_encoding TO 'utf8';"
!sudo -u postgres psql -c "ALTER ROLE ai SET default_transaction_isolation TO 'read committed';"
!sudo -u postgres psql -c "ALTER ROLE ai SET timezone TO 'UTC';"
!sudo -u postgres psql -c "GRANT ALL PRIVILEGES ON DATABASE ai TO ai;"

In [ ]:
from sqlalchemy import create_engine, text
import os

db_url = "postgresql+psycopg://ai:ai@localhost:5432/ai"
engine = create_engine(db_url)

try:
    print("Attempting to create PGVector extension as superuser...")
    # Execute CREATE EXTENSION command as postgres superuser
    # Connect to the 'ai' database to create the extension within it
    os.system('sudo -u postgres psql -d ai -c "CREATE EXTENSION IF NOT EXISTS vector;"')
    print("PGVector extension creation command executed by superuser.")
except Exception as e:
    print(f"Error during superuser extension creation: {e}")

with engine.connect() as conn:
    # No need to execute CREATE EXTENSION again with limited 'ai' user
    print("✅ PGVector extension is now enabled (or was already enabled)!")

In [ ]:
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores.pgvector import PGVector
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from agno.agent import Agent
from agno.models.groq import Groq


class DocumentQA:
    def __init__(self):
        self.db_url = "postgresql+psycopg://ai:ai@localhost:5432/ai"

        # Embeddings
        self.embedder = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-MiniLM-L6-v2"
        )

        # LLM
        self.chat_model = Groq(id="llama-3.3-70b-versatile")

        self.vectorstore = None
        self.agent = None

    def load_pdf_url(self, url: str, collection_name="docs"):
        try:
            print(f"Downloading PDF...")

            pdf_path = "/content/doc.pdf"
            response = requests.get(url)
            with open(pdf_path, "wb") as f:
                f.write(response.content)

            print("Loading PDF...")
            loader = PyPDFLoader(pdf_path)
            docs = loader.load()

            print(f"Loaded {len(docs)} pages")

            # Split
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200
            )
            texts = splitter.split_documents(docs)

            print(f"Split into {len(texts)} chunks")

            # Store in PGVector
            self.vectorstore = PGVector.from_documents(
                documents=texts,
                embedding=self.embedder,
                collection_name=collection_name,
                connection_string=self.db_url,
                pre_delete_collection=True
            )

            # Agent
            self.agent = Agent(
                model=self.chat_model
            )

            print("✅ Knowledge base ready!")

        except Exception as e:
            print(f"❌ Error: {e}")

    def ask(self, question: str):
        try:
            docs = self.vectorstore.similarity_search(question, k=5)

            context = "\n".join([d.page_content for d in docs])

            prompt = f"""
            Use ONLY the context below to answer.

            Context:
            {context}

            Question:
            {question}
            """

            response = self.agent.run(prompt)

            answer = response.content if hasattr(response, "content") else response
            print("\n✅ Answer:\n")
            print(answer)

        except Exception as e:
            print(f"❌ Error: {e}")

In [ ]:
 RAG_qa = DocumentQA()

In [ ]:
RAG_qa.load_pdf_url(
    "https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2024.pdf"
)

In [ ]:
RAG_qa.ask("Key points in this report? Give in 5 bullets")

In [ ]:
RAG_qa.ask("Name of company and their work")

In [ ]:
RAG_qa.ask("what is the content discussed")

In [ ]:
RAG_qa.ask("What products do Apple intend to create")

In [ ]:
RAG_qa.ask("What are the health impacts of the product")

In [ ]:
RAG_qa.ask("Describe yourself as an agent")